In [1]:
import gymnasium as gym
import copy
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import rl_mind.envs
from dataclasses import dataclass, replace
from scipy import stats
from torch import Tensor
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm
from rl_mind.core import Action, Actor
from rl_mind.nn import build_mlp, soft_update
from rl_mind.env import VecEnv
from rl_mind.data import ReplayBuffer, Transitions
from rl_mind.collectors import TransitionCollector
from rl_mind.evaluation import Evaluator
from rl_mind.notebook import run_directory, setup_tensorboard, silence_known_warnings

from src.rl_miniproject_lunarlander import run_td3, run_ddpg, TD3Config, DDPGConfig, monte_carlo_Q, ForceInitialStateWrapper

In [ ]:
@dataclass(frozen=True)
class DDPG_custom_config(DDPGConfig):
    #Modifier comme on veut ici
    seed: int = 1
    
ddpg_cfg = DDPG_custom_config()
ddpg = run_ddpg(ddpg_cfg)

  0%|          | 0/30000 [00:00<?, ?it/s]

In [4]:
ddpg.visualize_best()

In [ ]:
@dataclass(frozen=True)
class TD3_custom_config(TD3Config):
    #Modifier comme on veut ici
    seed: int = 1

td3_cfg = TD3_custom_config()esti
td3 = run_td3(td3_cfg)

In [7]:
td3.visualize_best()

examples utilisation monte carlo

In [32]:
@dataclass(frozen=True)
class Config16(DDPGConfig):
    #Modifier comme on veut ici
    monte_carlo_n_envs : int = 16

cfg = Config16()

In [ ]:
forced_initial_env = VecEnv(
    env_name=cfg.env_name,
    num_envs=cfg.monte_carlo_n_envs,
    seed=cfg.monte_carlo_seed,
    wrappers=[ForceInitialStateWrapper,]
)

env = VecEnv(ddpg_cfg.env_name, 2, seed=1)
obs = env.reset()
actor=ddpg.best_actor

In [ ]:
monte_carlo_Q(forced_initial_env, obs[0], actor.act(obs)[0], actor, 10000, cfg.mc_n_episodes)


tensor(6.3120)
tensor(5.3379)


overestimation bias

We measure the critics' overestimation bias by comparing their Q(s, a) estimates with Monte Carlo returns: first DDPG vs TD3 (Figure 1), then the effect of LayerNorm in the critic (Figure 2).

In [2]:
@dataclass(frozen=True)
class DDPG_custom_config(DDPGConfig):
    seed: int = 1
    mc_interval: int = 2_000      # one check every 2000 env steps
    mc_n_samples: int = 10         # (s, a) pairs taken from the batch
    mc_n_episodes: int = 100      # rollouts per pair
    monte_carlo_n_envs: int = 16  # parallel envs for the rollouts

@dataclass(frozen=True)
class TD3_custom_config(TD3Config):
    seed: int = 1
    mc_interval: int = 2_000
    mc_n_samples: int = 10
    mc_n_episodes: int = 100
    monte_carlo_n_envs: int = 16

In [3]:
ddpg = run_ddpg(DDPG_custom_config())
td3 = run_td3(TD3_custom_config())

  0%|          | 0/30000 [00:00<?, ?it/s]

  0%|          | 0/30000 [00:00<?, ?it/s]

In [ ]:
import numpy as np
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

def read_scalar(run_dir, tag):
    """Steps and values of a scalar logged in tensorboard"""
    ea = EventAccumulator(str(run_dir)); ea.Reload()
    ev = ea.Scalars(tag)
    return np.array([e.step for e in ev]), np.array([e.value for e in ev])

def plot_mean_se(ax, run_dir, tag, **kwargs):
    """Plots a Monte Carlo check (mean over the sampled (s, a) pairs) with a ± standard error band"""
    steps, mean = read_scalar(run_dir, tag)
    _, se = read_scalar(run_dir, f"{tag}_se")
    line, = ax.plot(steps, mean, **kwargs)
    ax.fill_between(steps, mean - se, mean + se, color=line.get_color(), alpha=0.2)
    return line

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
for name, ev, critic, critic_label in [("DDPG", ddpg, "critic", "critic"), ("TD3", td3, "critic_min", "min of the 2 critics")]:
    line = plot_mean_se(ax1, ev.run_dir, f"monte_carlo/q_{critic}", label=f"{name}: {critic_label}")
    plot_mean_se(ax1, ev.run_dir, "monte_carlo/q_mc", ls="--", color=line.get_color(), label=f"{name}: Monte Carlo")
    plot_mean_se(ax2, ev.run_dir, f"monte_carlo/bias_{critic}", color=line.get_color(), label=name)

ax1.set(xlabel="environment steps", ylabel="Q(s, a)  (discounted return, γ = 0.98)",
        title="(a) Critic estimate vs Monte Carlo return")
ax1.legend(fontsize=9)
ax2.axhline(0, color="gray", lw=0.8, ls=":")
ax2.set(xlabel="environment steps", ylabel="$Q_{critic} - Q_{MC}$",
        title="(b) Estimation bias (> 0: overestimation)")
ax2.legend(fontsize=9)
fig.suptitle("Figure 1 — Critic overestimation during training: DDPG vs TD3 (seed 1)")
fig.tight_layout()

**Figure 1 — Critic overestimation during training, DDPG vs TD3 (seed 1).** Every 2000 environment steps, 10 (s, a) pairs are drawn from the training batch. For each pair, the critic's estimate Q(s, a) is compared with a Monte Carlo estimate (mean discounted return of 100 episodes started from s with action a, then following the current deterministic actor). For TD3, the critic estimate is the min of its two critics, which is the value TD3 uses in its targets. **(a)** Critic estimate (solid) and Monte Carlo estimate (dashed). **(b)** Bias = critic − Monte Carlo; a value above 0 means the critic overestimates. Lines show the mean over the 10 pairs; shaded bands show ± one standard error of that mean, computed over the 10 pairs. Results come from a single training seed.

*To retain:* TODO, write the conclusion once the figure is produced (e.g. whether DDPG's bias stays above TD3's).

In [5]:
from dataclasses import replace

mc = dict(mc_interval=2_000, mc_n_samples=5, mc_n_episodes=100, monte_carlo_n_envs=16)
seeds = [1, 2, 3]

runs = {}  # (algo, layer_norm, seed) -> evaluator
for seed in seeds:
    for ln in [False, True]:
        runs["DDPG", ln, seed] = run_ddpg(DDPGConfig(seed=seed, critic_layer_norm=ln, **mc))
        runs["TD3", ln, seed] = run_td3(TD3Config(seed=seed, critic_layer_norm=ln, **mc))


TypeError: DDPGConfig.__init__() got an unexpected keyword argument 'critic_layer_norm'

In [ ]:
colors = {False: "C0", True: "C1"}
labels = {False: "without LayerNorm", True: "with LayerNorm"}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, algo, critic in [(axes[0], "DDPG", "critic"), (axes[1], "TD3", "critic_min")]:
    late_bias = {}
    for ln in [False, True]:
        curves = [read_scalar(runs[algo, ln, s].run_dir, f"monte_carlo/bias_{critic}") for s in seeds]
        steps = curves[0][0]
        b = np.stack([values for _, values in curves])  # [n_seeds, n_checks]
        for values in b:  # individual seeds
            ax.plot(steps, values, color=colors[ln], lw=0.6, alpha=0.35)
        mean, se = b.mean(0), b.std(0, ddof=1) / np.sqrt(len(seeds))
        ax.plot(steps, mean, color=colors[ln], lw=2, label=labels[ln])
        ax.fill_between(steps, mean - se, mean + se, color=colors[ln], alpha=0.2)
        # per-seed mean bias over the second half of training, used for the test
        late_bias[ln] = b[:, len(steps) // 2:].mean(1)

    test = stats.ttest_ind(late_bias[True], late_bias[False], equal_var=False)
    print(f"{algo}: late bias without LN {late_bias[False].round(1)}, with LN {late_bias[True].round(1)}, "
          f"Welch t = {test.statistic:.2f}, p = {test.pvalue:.3f}")
    ax.text(0.02, 0.03, f"Welch t-test (2nd half of training):\nt = {test.statistic:.2f}, p = {test.pvalue:.3f}",
            transform=ax.transAxes, fontsize=9, bbox=dict(boxstyle="round", fc="white", alpha=0.8))
    ax.axhline(0, color="gray", lw=0.8, ls=":")
    ax.set(title=f"({'a' if algo == 'DDPG' else 'b'}) {algo}", xlabel="environment steps")
    ax.legend(loc="upper right", fontsize=9)
axes[0].set_ylabel("$Q_{critic} - Q_{MC}$  (> 0: overestimation)")
fig.suptitle(f"Figure 2 — Effect of LayerNorm in the critic on the overestimation bias ({len(seeds)} seeds)")
fig.tight_layout()

**Figure 2 — Effect of LayerNorm in the critic on the overestimation bias.** Bias = critic estimate − Monte Carlo estimate of Q(s, a), measured every 2000 environment steps on 5 (s, a) pairs from the training batch (100 Monte Carlo episodes per pair). A value above 0 means the critic overestimates. For TD3, the critic estimate is the min of its two critics. **(a)** DDPG, **(b)** TD3, each trained without (blue) and with (orange) LayerNorm in the critic's hidden layers. Thin lines show individual seeds; thick lines show the mean over 3 seeds; shaded bands show ± one standard error of the mean over the seeds. In each panel, the box reports a Welch t-test comparing the per-seed mean bias over the second half of training, with vs without LayerNorm (n = 3 seeds per group, so the test has low power).

*To retain:* TODO, write the conclusion once the figure is produced.